##**This script sets up a shared global variable (shared_counter) and fires off two separate threads to run a loop 10,000 times each**

In [ ]:
import threading
import time

# This is our unprotected shared data
shared_counter = 0

def increment_counter():
    global shared_counter
    for _ in range(10000):
        # Read the current value
        current_value = shared_counter

        # Introduce a microscopic delay to force the OS scheduler
        # to switch threads right here, causing a race condition!
        time.sleep(0.000001)

        # Modify and write back
        shared_counter = current_value + 1

# Create two worker threads targeting the same function
thread_1 = threading.Thread(target=increment_counter)
thread_2 = threading.Thread(target=increment_counter)

print("Starting calculation...")
# Start both threads racing against each other
thread_1.start()
thread_2.start()

# Wait for both threads to completely finish
thread_1.join()
thread_2.join()

print(f"Final Counter Value: {shared_counter}")
print(f"Expected Value: 20000")

Starting calculation...
Final Counter Value: 10000
Expected Value: 20000


##**Fixing it with a threading.lock() and verifying that the count is always 20,000.**

In [ ]:
import threading
import time

# 1. Initialize the shared counter and our thread lock
shared_counter = 0
counter_lock = threading.Lock()

def safe_increment_counter():
    global shared_counter
    for _ in range(10000):
        # 2. Acquire the lock before entering the critical section
        with counter_lock:
            current_value = shared_counter
            time.sleep(0.000001)  # The sleep delay cannot break us now!
            shared_counter = current_value + 1
        # 3. The lock is automatically released here when the block ends

# Create two worker threads targeting our new safe function
thread_1 = threading.Thread(target=safe_increment_counter)
thread_2 = threading.Thread(target=safe_increment_counter)

print("Starting safe calculation...")
thread_1.start()
thread_2.start()

thread_1.join()
thread_2.join()

print(f"Final Safe Counter Value: {shared_counter}")
print(f"Expected Value: 20000")

Starting safe calculation...
Final Safe Counter Value: 20000
Expected Value: 20000


In [ ]:
# WHERE THE RACE CONDITION OCCURRED:
# The code "shared_counter = current_value + 1" takes three separate steps:
# 1. Read: Take the current value of shared_counter from main memory (RAM).
# 2. Modify: Add 1 to that number inside the CPU.
# 3. Write: Save the new total back out to main memory (RAM).
# Because of time.sleep, the OS pauses Thread 1 right after Step 1 (Read) but before Step 3 (Write).
# Thread 2 wakes up and reads the exact same stale data (e.g., 0). Both threads calculate
# the same value and write it to the exact same spot, completely overwriting each other's work
# and cutting the final count in half.

# WHY THE LOCK FIXED IT:
# Adding "with counter_lock:" introduces Mutual Exclusion. It turns all three steps
# (Read, Modify, Write) into an unbreakable package deal.
# When Thread 1 grabs the lock key, the critical section is locked down. If the OS tries to
# switch to Thread 2, Thread 2 sees the key is taken and is blocked instantly, sleeping until
# Thread 1 completely finishes saving its work. This forces threads to wait in a clean line,
# guaranteeing a perfect final calculation every single time.